# 58 — Existing models on adversarial pairs

**Purpose:** Evaluate **saved checkpoints** on anchor vs counterfactual texts from **56**, using **audit context** from **57** as optional QC. **No training** and **no pair regeneration**.

**Shared helper:** `notebooks/adversarial/adversarial_eval_common.py` → `run_adversarial_pair_model_eval`.

**Outputs per model:** `notebooks/results/adversarial_model_eval/<model_run_id>/` and `figures/adversarial_model_eval/<model_run_id>/`.

Set `ADV_MODEL_RUN_ID` to choose a single checkpoint, or set `ADV_EVAL_ALL_MODELS=1` to iterate all IDs registered in `MODEL_CHECKPOINT_CANDIDATES` (skips missing checkpoints).


In [1]:
import os
import sys
from pathlib import Path

_CWD = Path.cwd().resolve()
_CANDS = [_CWD, _CWD / "notebooks", _CWD.parent]
ADV_DIR = next((p / "adversarial" for p in _CANDS if (p / "adversarial" / "adversarial_eval_common.py").exists()), None)
if ADV_DIR is None:
    raise FileNotFoundError("adversarial_eval_common.py not found under notebooks/adversarial")
if str(ADV_DIR) not in sys.path:
    sys.path.insert(0, str(ADV_DIR))

MODEL_RUN_ID = os.environ.get("ADV_MODEL_RUN_ID", "bert_9classes_final")
print("MODEL_RUN_ID:", MODEL_RUN_ID)


MODEL_RUN_ID: bert_9classes_final


In [2]:
import json
import os

from adversarial_eval_common import run_adversarial_pair_model_eval
from e07_english_eval_common import resolve_repo_root, MODEL_CHECKPOINT_CANDIDATES

repo = resolve_repo_root()
pairs_csv = repo / "notebooks" / "results" / "adversarial_resume_pairs" / "adversarial_resume_pairs.csv"

RUN_ALL = os.environ.get("ADV_EVAL_ALL_MODELS", "").strip() in ("1", "true", "True")
ids = list(MODEL_CHECKPOINT_CANDIDATES.keys()) if RUN_ALL else [MODEL_RUN_ID]

all_summaries = []
for mid in ids:
    print("---", mid, "---")
    try:
        s = run_adversarial_pair_model_eval(
            mid,
            pairs_csv=pairs_csv,
            batch_size=8,
            max_length=256,
            results_subdir="adversarial_model_eval",
            figures_subdir="adversarial_model_eval",
        )
        all_summaries.append(
            {
                "model_run_id": mid,
                "flip_rate": s.get("flip_rate"),
                "mean_delta_prob_true_class": s.get("mean_delta_prob_true_class"),
                "n_pairs_evaluated": s.get("n_pairs_evaluated"),
            }
        )
    except FileNotFoundError as e:
        print("Skip (missing checkpoint or pairs):", e)
    except Exception as e:
        print("Skip (error):", mid, e)

print(json.dumps(all_summaries, indent=2))


/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- bert_9classes_final ---


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 776.98it/s, Materializing param=classifier.weight]                                      


[
  {
    "model_run_id": "bert_9classes_final",
    "flip_rate": 0.02403846153846154,
    "mean_delta_prob_true_class": 0.0001972593308892101,
    "n_pairs_evaluated": 208
  }
]


## Key takeaways

1. **Stability signal:** **Flip rate** and **mean Δ probability for the true class** summarize robustness to approved minimal edits; inspect `metrics.json` for **per-attribute** and **per-class** slices.
2. **Sensitivity:** Higher flip rate on specific `attribute_edited` values highlights **proxy sensitivity** (e.g., geography or pronouns) complementary to **fairness** analyses and **city-swap** stress tests from alignment work (32).
3. **Next steps:** Compare runs across `MODEL_RUN_ID` values using separate notebook executions or environment overrides; keep **deterministic** output paths for parallel launches.
